## Data Manipulation and Exploration with Spark

### Set up the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Connect to your Azure ML Workspace

In [ ]:
import azureml.core
from azureml.core import Workspace

# Load the workspace from the saved config file
ws = Workspace.from_config()
print('Ready to use Azure ML {} to work with {}'.format(azureml.core.VERSION, ws.name))

### Get the Default Datastore

In [ ]:
# Get the default datastore
default_ds = ws.get_default_datastore()

# Enumerate all datastores, indicating which is the default
for ds_name in ws.datastores:
    print(ds_name, "- Default =", ds_name == default_ds.name)

### Upload Data to the Datastore

In [ ]:
import requests

url = "https://raw.githubusercontent.com/kuljotSB/AI-300/main/Data/Sales/2019.csv"

r = requests.get(url)

files = ["2019.csv", "2020.csv", "2021.csv"]
for file in files:
    with open(file, "wb") as f:
        f.write(r.content)

    default_ds.upload_files(
        files=[file],
        target_path="sales-data/",
        overwrite=True,
        show_progress=True
    )

### Create a Tabular Dataset from the Uploaded Data

In [ ]:
from azureml.core import Dataset

# Get the default datastore
default_ds = ws.get_default_datastore()

# Create a tabular dataset
tab_data_set = Dataset.Tabular.from_delimited_files(
    path=(default_ds, 'sales-data/*.csv')
)

# Convert to Spark DataFrame
spark_df = tab_data_set.to_spark_dataframe()

# Show rows
spark_df.show(20)

### Defining schema for the dataframe

In [ ]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Convert Azure ML TabularDataset to Spark DataFrame
df = tab_data_set.to_spark_dataframe()

# Apply schema / casting
df = df.select(
    col("SalesOrderNumber").cast(StringType()),
    col("SalesOrderLineNumber").cast(IntegerType()),
    col("OrderDate").cast(DateType()),
    col("CustomerName").cast(StringType()),
    col("Email").cast(StringType()),
    col("Item").cast(StringType()),
    col("Quantity").cast(IntegerType()),
    col("UnitPrice").cast(FloatType()),
    col("Tax").cast(FloatType())
)

df.show(100)

### Query Data using Spark SQL

In [ ]:
df.createOrReplaceTempView("salesorders")
spark_df = spark.sql("SELECT * FROM salesorders")
display(spark_df)

In [ ]:
sqlQuery = "SELECT CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear, \
               SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue \
        FROM salesorders \
        GROUP BY CAST(YEAR(OrderDate) AS CHAR(4)) \
        ORDER BY OrderYear"
df_spark = spark.sql(sqlQuery)
df_spark.show()

### Using Matplotlib to visualize the data

In [ ]:
from matplotlib import pyplot as plt

# matplotlib requires a Pandas dataframe, not a Spark one
df_sales = df_spark.toPandas()
# Create a bar plot of revenue by year
plt.bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'])
# Display the plot
plt.show()

### Using Seaborn to visualize the data

In [ ]:
import seaborn as sns

# Clear the plot area
plt.clf()
# Create a bar chart
ax = sns.barplot(x="OrderYear", y="GrossRevenue", data=df_sales)
plt.show()